# Trabalho de APM - Classificação - G13

### 1. Importar bibliotecas

In [20]:
import warnings
warnings.filterwarnings("ignore")
warnings.simplefilter(action='ignore', category=FutureWarning)

import numpy as np
import pandas as pd

# Pré-processamento
from sklearn.preprocessing import MinMaxScaler

# Seleção de modelo
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

# Modelos
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

# Metricas
from sklearn.metrics import  accuracy_score, confusion_matrix

#import joblib

### 2. Macro definições

In [21]:
# Bases de dados
DF_VEICULOS = "Veiculos - Dados.csv"
DF_DIABETES = "Diabetes - Dados.csv"

# Novos dados
DADOS_NOVOS_VEICULOS = "Veiculos - Novos Casos.csv"
DADOS_NOVOS_DIABETES = "Diabetes - Novos Casos.csv"

# Seed Global
RAND_SEED = 202613 # Ano atual + número do Grupo
np.random.seed(RAND_SEED)

# Tamanho da base de teste: 30%
TEST_SIZE = 0.3

# Largura máxima das colunas a serem exibidas
pd.set_option('display.max_colwidth', None)

### 3. Funções locais

In [22]:
# Carrega DataFrame
def load_df(df_name, column_to_drop=""):
    """Função para carregar o csv e se informado, remover colunas"""
    try:
        df = pd.read_csv(df_name, sep=',')
        if column_to_drop:
            df = df.drop(column_to_drop, axis=1)
        return df
    
    except:
        print("Erro ao processar o arquivo", df_name)

In [23]:
# Treina e avalia uma lista de modelos
def treina_avalia(models, X_train, X_test, y_train, y_test, classes):
    """Treina e avalia uma lista de modelos, retorna um dataframe com os resultados"""
    trained_models = {} # Dicionario para guardar os treinos

    # loop para iterar entre os modelos
    for name, model in models.items():
    
        model.fit(X_train, y_train)  # Treina
    
        # Mostra os melhores paramentros do modelo com GridSearchCV
        parametros = "Valores default"
        if isinstance(model, GridSearchCV):
            print('Melhores parametros para o modelo', model.estimator.__class__.__name__)
            print(model.best_params_, '\n')
            parametros = model.best_params_
    
        y_pred = model.predict(X_test) # faz a predição
        
        trained_models[name] = {
            "Acuracia" : round(accuracy_score(y_test, y_pred), 2), # calcula a acurácia
            "Parametros" : parametros, # salva os parametros utilizados
            "Matriz_Confusao" : confusion_matrix(y_test, y_pred, labels=classes) # calcula a matriz de confusão
        } # salva no dicionario
        
    
    # Guarda em um Dataframe
    df_trained_models = pd.DataFrame(trained_models).T
    df_trained_models.index.name = "Tecnica"
    df_trained_models = df_trained_models.sort_values(
                        'Acuracia',
                        ascending=False)
    # Retorna um dataframe com o resultado
    return df_trained_models

### 4. Definição de Modelos

In [24]:
# KNN
param_grid_KNN = {
    'n_neighbors': [1, 3, 5, 7, 9]
}
KNN = GridSearchCV(KNeighborsClassifier(), param_grid_KNN, n_jobs= -1, cv=10, scoring='accuracy')

In [25]:
# RNA
param_grid_RNA = {
    'hidden_layer_sizes': [(10,), (30,), (100,)],
    'learning_rate_init': [0.1, 0.01, 0.001]
}
RNA_CV = GridSearchCV(MLPClassifier(random_state=RAND_SEED), param_grid_RNA, n_jobs= -1, cv=10, scoring='accuracy')

RNA_HO = MLPClassifier(random_state=RAND_SEED)

In [26]:
# SVM
param_grid_SVM = {
    'C': [1, 10, 50, 100],
    'gamma': ['auto', 'scale']  
}

SVM_CV = GridSearchCV(SVC(random_state=RAND_SEED), param_grid_SVM, n_jobs= -1, cv=10, scoring='accuracy')

SVM_HO = SVC(random_state=RAND_SEED)

In [27]:
# RainForest
param_grid_RF = {
    'n_estimators': list(range(50, 101, 10)),
    'max_depth': list(range(1, 21, 10)),
}
RF_CV = GridSearchCV(RandomForestClassifier(random_state=RAND_SEED), param_grid_RF, n_jobs= -1, cv=10, scoring='accuracy')

RF_DF = RandomForestClassifier(random_state=RAND_SEED)

In [28]:
models = {
    "KNN": KNN,
    "RNA_HOLD_OUT": RNA_HO,
    "RNA_CV": RNA_CV,
    "SVM_HOLD_OUT": SVM_HO,
    "SVM_CV": SVM_CV,
    "RF_DEFAULT": RF_DF,
    "RF_CV": RF_CV    
}

### 5. Carrega database de Veiculos e pré-processa

In [29]:
scaler = MinMaxScaler()

df = load_df(DF_VEICULOS, 'a')

y = df['tipo']
X = df.drop('tipo', axis = 1)

columns = list(X.columns)
classes = y.unique().tolist()

# Restaura os nomes das colunas
X = pd.DataFrame(X, columns=columns)

# Separa em treino, teste e indices
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RAND_SEED
)

# Preprocessa features
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### 6. Treina em diferentes modelos e armazena os resultados

In [30]:
resultado_veiculos = treina_avalia(models, X_train, X_test, y_train, y_test, classes )

Melhores parametros para o modelo KNeighborsClassifier
{'n_neighbors': 3} 

Melhores parametros para o modelo MLPClassifier
{'hidden_layer_sizes': (100,), 'learning_rate_init': 0.01} 

Melhores parametros para o modelo SVC
{'C': 100, 'gamma': 'scale'} 

Melhores parametros para o modelo RandomForestClassifier
{'max_depth': 11, 'n_estimators': 50} 



In [31]:
# Exibe resultado do Experimento com base de Veículos

print(' ')
print('###################')
print('Experimento - Veiculos')
display(resultado_veiculos)

 
###################
Experimento - Veiculos


,Acuracia,Parametros,Matriz_Confusao
Tecnica,,,
SVM_CV,0.85,"{'C': 100, 'gamma': 'scale'}","[[75, 0, 0, 0], [1, 49, 1, 19], [2, 0, 52, 0], [1, 13, 1, 40]]"
RNA_CV,0.81,"{'hidden_layer_sizes': (100,), 'learning_rate_init': 0.01}","[[74, 0, 0, 1], [1, 34, 2, 33], [1, 1, 52, 0], [1, 8, 0, 46]]"
RNA_HOLD_OUT,0.75,Valores default,"[[74, 0, 0, 1], [5, 31, 4, 30], [3, 0, 51, 0], [4, 15, 1, 35]]"
SVM_HOLD_OUT,0.75,Valores default,"[[75, 0, 0, 0], [4, 27, 5, 34], [4, 0, 50, 0], [3, 11, 2, 39]]"
RF_CV,0.74,"{'max_depth': 11, 'n_estimators': 50}","[[74, 1, 0, 0], [5, 34, 3, 28], [3, 0, 50, 1], [1, 23, 2, 29]]"
RF_DEFAULT,0.73,Valores default,"[[74, 1, 0, 0], [6, 32, 3, 29], [4, 0, 50, 0], [1, 22, 2, 30]]"
KNN,0.69,{'n_neighbors': 3},"[[63, 2, 5, 5], [5, 35, 6, 24], [2, 1, 49, 2], [3, 23, 1, 28]]"


In [32]:
# Exibe a matriz de confusão de cada modelo
for idx, row in resultado_veiculos.iterrows():
    print(' ')
    print(idx)
    cm = row["Matriz_Confusao"]
    temp_df = pd.DataFrame(
        cm,
        index=classes,
        columns=classes)
    display(temp_df)

 
SVM_CV


,van,saab,bus,opel
van,75,0,0,0
saab,1,49,1,19
bus,2,0,52,0
opel,1,13,1,40


 
RNA_CV


,van,saab,bus,opel
van,74,0,0,1
saab,1,34,2,33
bus,1,1,52,0
opel,1,8,0,46


 
RNA_HOLD_OUT


...

,van,saab,bus,opel
van,74,1,0,0
saab,5,34,3,28
bus,3,0,50,1
opel,1,23,2,29


 
RF_DEFAULT


,van,saab,bus,opel
van,74,1,0,0
saab,6,32,3,29
bus,4,0,50,0
opel,1,22,2,30


 
KNN


,van,saab,bus,opel
van,63,2,5,5
saab,5,35,6,24
bus,2,1,49,2
opel,3,23,1,28


### 7. Faz predição para dados novos de Veículos usando o melhor modelo

In [33]:
# obtem o melhor modelo
best_model_name = resultado_veiculos.index[0]
best_model = models[best_model_name]

# carrega novos dados
X_novo = load_df(DADOS_NOVOS_VEICULOS)

# Preprocessa features
X_novo = scaler.transform(X_novo)

# faz a predição
y_novo = best_model.predict(X_novo)
print(y_novo.tolist())


['van', 'van', 'saab']


### 8. Carrega database de Diabetes e pré-processa

In [34]:
scaler = MinMaxScaler()

df = load_df(DF_DIABETES, 'num')

y = df['diabetes']
X = df.drop('diabetes', axis = 1)

columns = list(X.columns)
classes = y.unique().tolist()

# Restaura os nomes das colunas
X = pd.DataFrame(X, columns=columns)

# Separa em treino, teste e indices
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RAND_SEED
)

# Preprocessa features
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### 9. Treina em diferentes modelos e armazena os resultados

In [35]:
resultado_diabetes = treina_avalia(models, X_train, X_test, y_train, y_test, classes )

Melhores parametros para o modelo KNeighborsClassifier
{'n_neighbors': 7} 

Melhores parametros para o modelo MLPClassifier
{'hidden_layer_sizes': (100,), 'learning_rate_init': 0.01} 

Melhores parametros para o modelo SVC
{'C': 10, 'gamma': 'scale'} 

Melhores parametros para o modelo RandomForestClassifier
{'max_depth': 11, 'n_estimators': 90} 



In [36]:
# Experimento com base de Diabetes
print(' ')
print('###################')
print('Experimento - Diabetes')
display(resultado_diabetes)

 
###################
Experimento - Diabetes


,Acuracia,Parametros,Matriz_Confusao
Tecnica,,,
RF_DEFAULT,0.78,Valores default,"[[49, 30], [21, 131]]"
RNA_HOLD_OUT,0.77,Valores default,"[[43, 36], [18, 134]]"
RF_CV,0.77,"{'max_depth': 11, 'n_estimators': 90}","[[47, 32], [21, 131]]"
RNA_CV,0.73,"{'hidden_layer_sizes': (100,), 'learning_rate_init': 0.01}","[[48, 31], [32, 120]]"
SVM_HOLD_OUT,0.73,Valores default,"[[39, 40], [23, 129]]"
SVM_CV,0.71,"{'C': 10, 'gamma': 'scale'}","[[40, 39], [29, 123]]"
KNN,0.7,{'n_neighbors': 7},"[[41, 38], [32, 120]]"


In [37]:
# Exibe a matriz de confusão de cada modelo
for idx, row in resultado_diabetes.iterrows():
    print(' ')
    print(idx)
    cm = row["Matriz_Confusao"]
    temp_df = pd.DataFrame(
        cm,
        index=classes,
        columns=classes)
    display(temp_df)

 
RF_DEFAULT


,pos,neg
pos,49,30
neg,21,131


 
RNA_HOLD_OUT


,pos,neg
pos,43,36
neg,18,134


 
RF_CV


...

,pos,neg
pos,39,40
neg,23,129


 
SVM_CV


,pos,neg
pos,40,39
neg,29,123


 
KNN


,pos,neg
pos,41,38
neg,32,120


### 10. Faz predição para dados novos de Diabetes usando o melhor modelo

In [38]:
# obtem o melhor modelo
best_model_name = resultado_diabetes.index[0]
best_model = models[best_model_name]

# carrega novos dados
X_novo = load_df(DADOS_NOVOS_DIABETES)

# Preprocessa features
X_novo = scaler.transform(X_novo)

# faz a predição
y_novo = best_model.predict(X_novo)
print(y_novo.tolist())


['neg', 'neg', 'pos']
